# 🗓️ 16 ~ 17일차 스터디 노트북 — 8퀸 문제

**오늘 범위**: 05-4 8퀸 문제 → 행과 열 → 분기 작업 → 한정 작업 → 백트래킹(flag의 True/False) → 대각선 인덱스(`i+j`, `i-j+7`) → 분기 한정법

*(05장 재귀 알고리즘 완주!)*

## 난이도 태그
- 🟢 **기본** — 전원 필수
- 🟡 **표준** — 팀 목표선
- 🔴 **심화** — 도전

## 유형 태그
**[손]** 손으로 그리기 · **[빈칸]** 빈칸 채우기 · [예측] 실행 전 결과 맞히기 · [구현] 코드 완성 · [디버깅] 버그 찾기 · [설명] 왜인지 서술 · [실험] 직접 찍어보기

## 🖐️ 오늘의 진행 방식 (중요!)
**손 → 머리 → 코드** 순서로 간다:
1. **1부 [손으로 그리기]** — 체스판에 직접 퀸을 그려보며 규칙을 몸으로 익히기
2. **2부 [재귀 과정 이해]** — `flag`가 켜지고 꺼지는 과정을 손으로 추적
3. **3부 [코드로 옮기기]** — 이제야 코드를 짠다

**1부를 건너뛰고 코드부터 보면 절대 이해 안 돼.** 꼭 순서대로 가자.

> 📁 아래 "부록: 코드 4형제" 셀을 **먼저 실행**해.

---

## 🔁 [Remind] 워밍업 — 15일차 하노이 복습

하노이와 8퀸은 교재가 나란히 놓은 이유가 있어. 둘 다 **"큰 문제를 작은 문제로 쪼개는"** 같은 전략이야.

### R-1. 🟢 [설명] 하노이와 8퀸의 공통점

교재 212p: "하노이의 탑이나 8퀸 문제처럼 큰 문제를 작은 문제로 분할하고, 작은 문제 풀이법을 결합하여 전체 풀이법을 얻는 방법을 **분할 정복법(divide and conquer)**이라고 합니다."

| | 하노이 (15일차) | 8퀸 (오늘) |
|---|---|---|
| 큰 문제 | 원반 n개 옮기기 | 퀸 8개 배치하기 |
| 쪼개는 단위 | **①____** | **②____** |
| 재귀 호출 | `move(no-1, ...)` | **③____** |
| 기저 조건 | `no == 1` | **④____** |

*(①~④ 채우기. ②는 "한 번에 몇 개를 처리하는가"를 생각해봐)*

### R-2. 🟡 [설명] 15일차 "믿음의 도약"이 8퀸에도 있나?

15일차에 이런 얘기를 했어:
> `move(no-1, ...)`이 내부적으로 어떻게 동작하는지 **몰라도** "알아서 해줄 거야"라고 믿고 짤 수 있다 = **믿음의 도약(leap of faith)**

8퀸의 `set(i)`에도 같은 게 있을까?

```python
def set(i):
    for j in range(8):
        if (놓을 수 있으면):
            pos[i] = j
            flag[j] = True
            set(i + 1)      # ← 여기
            flag[j] = False
```

- `set(i+1)`을 호출할 때, 우리는 그 안에서 무슨 일이 일어나는지 알아야 할까?
- `set(i)`가 책임지는 범위는 정확히 어디까지야?

*(여기에 답 작성)*

---
## 📦 부록 — 8퀸 코드 4형제

이 셀을 **먼저 실행**해. 오늘 만든 4개 파일이 전부 들어있어. (`print` 대신 결과를 리스트로 모으도록 살짝 바꿨어 — 1,677만 줄을 출력하면 노트북이 멈추거든)

In [ ]:
import io, contextlib

N = 8

# ---------------------------------------------------------
# 1단계 (8queen_b.py): 분기만 — 각 열에 퀸 1개씩, 그 외 규칙 없음
# ---------------------------------------------------------
def count_stage1():
    '''실습 5-7: 조합 개수만 세기 (실제 출력하면 1,677만 줄!)'''
    pos = [0] * N
    cnt = 0
    def put():
        nonlocal cnt
        cnt += 1
    def set_(i):
        for j in range(N):
            pos[i] = j
            if i == N - 1:
                put()
            else:
                set_(i + 1)
    set_(0)
    return cnt

# ---------------------------------------------------------
# 2단계 (8queen_bb.py): flag로 "같은 행 금지" 한정 추가
# ---------------------------------------------------------
def solve_stage2():
    '''실습 5-8: 행 중복만 막음'''
    pos = [0] * N
    flag = [False] * N
    sols = []
    def put():
        sols.append(list(pos))
    def set_(i):
        for j in range(N):
            if not flag[j]:
                pos[i] = j
                if i == N - 1:
                    put()
                else:
                    flag[j] = True
                    set_(i + 1)
                    flag[j] = False
    set_(0)
    return sols

# ---------------------------------------------------------
# 3단계 (8queen.py): 대각선까지 — 진짜 8퀸 문제 해결!
# ---------------------------------------------------------
def solve_stage3():
    '''실습 5-9: 행 + 양방향 대각선 전부 한정'''
    pos = [0] * N
    flag_a = [False] * N        # 행
    flag_b = [False] * (2*N-1)  # / 방향 대각선
    flag_c = [False] * (2*N-1)  # \ 방향 대각선
    sols = []
    def put():
        sols.append(list(pos))
    def set_(i):
        for j in range(N):
            if (not flag_a[j]
                and not flag_b[i + j]
                and not flag_c[i - j + (N-1)]):
                pos[i] = j
                if i == N - 1:
                    put()
                else:
                    flag_a[j] = flag_b[i + j] = flag_c[i - j + (N-1)] = True
                    set_(i + 1)
                    flag_a[j] = flag_b[i + j] = flag_c[i - j + (N-1)] = False
    set_(0)
    return sols


# ---------------------------------------------------------
# 체스판 그리기 헬퍼 (8queen2.py의 put() 방식)
# ---------------------------------------------------------
def draw(pos, mark='♛', empty='□'):
    '''pos 배열을 체스판으로 출력. pos[i] = j 는 i열 j행에 퀸'''
    n = len(pos)
    print('    ' + ' '.join(str(i) for i in range(n)) + '  ← 열(i)')
    for j in range(n):
        row = ' '.join(mark if pos[i] == j else empty for i in range(n))
        print(f'  {j} {row}')
    print('  ↑ 행(j)')

def is_valid(pos):
    '''서로 공격하지 않는 배치인지 검사'''
    n = len(pos)
    for a in range(n):
        for b in range(a + 1, n):
            if pos[a] == pos[b]:
                return False, f'{a}열과 {b}열이 같은 행({pos[a]})'
            if abs(pos[a] - pos[b]) == abs(a - b):
                return False, f'{a}열과 {b}열이 대각선으로 공격'
    return True, '유효한 배치!'


def solve_n(n):
    '''n×n 보드의 모든 해를 구함'''
    pos = [0] * n
    fa = [False] * n
    fb = [False] * (2*n-1)
    fc = [False] * (2*n-1)
    sols = []
    def set_(i):
        for j in range(n):
            if not fa[j] and not fb[i+j] and not fc[i-j+(n-1)]:
                pos[i] = j
                if i == n - 1:
                    sols.append(list(pos))
                else:
                    fa[j] = fb[i+j] = fc[i-j+(n-1)] = True
                    set_(i + 1)
                    fa[j] = fb[i+j] = fc[i-j+(n-1)] = False
    set_(0)
    return sols

print('준비 완료 — draw(), solve_n(), is_valid(), solve_stage2(), solve_stage3() 사용 가능')


---
## 📖 오늘의 핵심 개념

### 1. 8퀸 문제란?
> 8개의 퀸이 **서로 공격하여 잡을 수 없도록** 8×8 체스판에 배치하세요.

퀸은 **가로, 세로, 대각선** 어디든 직선으로 이동해서 상대를 잡을 수 있어 (장기의 차 + 밀 합친 것).

- 19세기 수학자 **카를 F. 가우스**가 **오답**을 발표한 것으로 유명해 😅
- 정답은 총 **92가지**

### 2. 행(行)과 열(列) — 오늘 제일 헷갈렸던 것 🎯

> **가로로 나란히 있는 한 팀 = 행 / 세로로 나란히 있는 한 팀 = 열**

여기에 하나 더:

| | 팀(줄)의 모양 | 그 팀에 번호 매기는 방향 |
|---|---|---|
| **행** | 가로 (→ 한 팀) | 위 → 아래 |
| **열** | 세로 (↓ 한 팀) | 왼쪽 → 오른쪽 |

**팀의 생김새**와 **번호 세는 방향**은 서로 **수직**이야. 이걸 하나로 뭉치면 계속 꼬여.

### 3. `pos` 배열의 의미

```python
pos[i] = j    # i열에 배치한 퀸의 위치가 j행
```

- **`i` (배열 인덱스)** = **열** 번호 → 왼쪽부터 오른쪽으로
- **`pos[i]` (배열 값)** = **행** 번호 → 그 열 안에서 위에서부터

배열 하나로 8개 퀸의 위치를 전부 표현할 수 있는 이유: **규칙 1(각 열에 퀸 1개만)** 덕분에 "열 하나당 행 하나"가 확정되니까.

### 4. 조합의 수가 줄어드는 3단계 🔥

| 단계 | 적용 규칙 | 조합 수 | 파일 |
|---|---|---|---|
| 아무 제약 없음 | — | 178,462,987,637,760 | — |
| **분기만** | 규칙 1: 각 **열**에 1개 | **8⁸ = 16,777,216** | `8queen_b.py` |
| **+ 행 한정** | 규칙 2: 각 **행**에 1개 | **8! = 40,320** | `8queen_bb.py` |
| **+ 대각선 한정** | 대각선에도 1개 | **92** ✓ | `8queen.py` |

### 5. 분기 작업 vs 한정 작업

- **분기(branching)**: 가지를 뻗듯 모든 경우를 나열하는 것 → `set(i+1)` 재귀 호출
- **한정(bounding)**: 필요 없는 가지를 **미리 쳐내는** 것 → `if not flag[j]`
- 둘을 합친 게 **분기 한정법(branching and bounding method)**

### 6. flag의 True/False — 백트래킹 🎯

```python
flag[j] = True      # ① "j행은 지금 내가 쓰는 중" 예약
set(i + 1)          # ② 이 선택으로 갈 수 있는 데까지 전부 탐색
flag[j] = False     # ③ 다 끝났으니 예약 반납
```

**왜 반납해야 하나**: `for j` 루프는 아직 살아있어. 다음 후보 행(`j+1`)을 시도할 땐 방금 쓴 `j`행이 다시 비어야 해.

**언제 반납되나**: `set(i+1)`이 **완전히 끝나서 돌아온 뒤**에만. 함수 호출은 끝나야 다음 줄로 가니까(11일차 스택 LIFO), **더 깊은 열이 실행되는 동안에는 `flag[j]`가 계속 `True`인 게 보장**돼.

이 "시도 → 끝까지 파본다 → 원상복구 → 다음 시도" 패턴을 **백트래킹(backtracking)**이라고 해.

### 7. 대각선 인덱스 — `i+j`와 `i-j+7` 🎯

**`/` 방향 대각선**: 오른쪽 위로 이동 = 열 +1, 행 −1
```
(i+1) + (j-1) = i + j     ← 합이 변하지 않음!
```
→ 같은 `/` 대각선 위의 모든 칸은 **`i+j`가 같다** → `flag_b[i+j]`

**`\` 방향 대각선**: 오른쪽 아래로 이동 = 열 +1, 행 +1
```
(i+1) - (j+1) = i - j     ← 차가 변하지 않음!
```
→ 같은 `\` 대각선은 **`i-j`가 같다**. 근데 `i-j`는 −7~7이라 음수가 나와서 배열 인덱스로 못 써 → **`+7`을 더해** 0~14로 밀어줌 → `flag_c[i-j+7]`

**`flag_b`, `flag_c`가 15칸인 이유**: 대각선 개수가 `2×8-1 = 15`개니까.

---
# 🖐️ 1부 — 손으로 그려보기

**코드는 아직 보지 마.** 종이나 아래 표에 직접 그려보면서 감을 잡자.

### 1. 🟢 [손] 퀸이 공격하는 칸 칠하기

아래 체스판의 `♛` 위치(3열 3행)에서 퀸이 **공격할 수 있는 칸**을 전부 `X`로 채워봐.

```
    0 1 2 3 4 5 6 7   ← 열
  0 □ □ □ □ □ □ □ □
  1 □ □ □ □ □ □ □ □
  2 □ □ □ □ □ □ □ □
  3 □ □ □ ♛ □ □ □ □
  4 □ □ □ □ □ □ □ □
  5 □ □ □ □ □ □ □ □
  6 □ □ □ □ □ □ □ □
  7 □ □ □ □ □ □ □ □
  ↑ 행
```

*(다 채웠으면 아래 실행해서 확인)*

In [ ]:
def show_attack(qi, qj, n=8):
    '''(qi열, qj행)의 퀸이 공격하는 칸을 X로 표시'''
    print('    ' + ' '.join(str(i) for i in range(n)) + '  ← 열')
    for j in range(n):
        row = []
        for i in range(n):
            if i == qi and j == qj:
                row.append('♛')
            elif i == qi or j == qj or abs(i - qi) == abs(j - qj):
                row.append('X')
            else:
                row.append('□')
        print(f'  {j} ' + ' '.join(row))
    print('  ↑ 행')

show_attack(3, 3)


### 2. 🟢 [손] 3×3 보드에 3퀸 배치해보기

**8×8은 너무 크니까 3×3부터.** 아래 판에 퀸 3개를 서로 공격 안 하게 놓아봐.

```
    0 1 2   ← 열
  0 □ □ □
  1 □ □ □
  2 □ □ □
  ↑ 행
```

- 놓을 수 있었어? 몇 가지 방법이 있었어?
- 못 놓았다면, **왜** 불가능한지 설명해봐.

*(여기에 답 작성 → 아래로 확인)*

In [ ]:
for n in range(1, 9):
    s = solve_n(n)
    print(f'{n}×{n} 보드: {len(s):>3}가지')


### 3. 🟢 [손] 4×4 보드 직접 풀기

3×3은 답이 없었지? **4×4는 답이 있어.** 직접 찾아봐.

```
    0 1 2 3   ← 열
  0 □ □ □ □
  1 □ □ □ □
  2 □ □ □ □
  3 □ □ □ □
  ↑ 행
```

**힌트**: 0열부터 차례로 놓되, 한 열에 퀸 1개씩. 막히면 되돌아가서 다른 행을 시도해봐 — **이게 바로 백트래킹이야!**

찾았으면 `pos` 배열로 적어봐: `pos = [__, __, __, __]`

In [ ]:
sols4 = solve_n(4)
print(f'4×4의 해: {len(sols4)}가지\n')
for s in sols4:
    print(f'pos = {s}')
    draw(s)
    print()


### 4. 🟡 [손/빈칸] 대각선 번호 매기기 🔥

7번 개념에서 배운 `i+j`를 **직접 채워봐.** 각 칸에 `i+j` 값을 적는 거야. (4×4로 축소)

```
        열(i)→
        0    1    2    3
  0   [ 0 ][ 1 ][__ ][__ ]
  1   [ 1 ][__ ][__ ][__ ]
  2   [__ ][__ ][__ ][__ ]
  3   [__ ][__ ][__ ][ 6 ]
  ↑행(j)
```

다 채웠으면:
- **같은 숫자끼리 이어보면** 어떤 모양이 나와? (`/` 방향? `\` 방향?)
- 4×4에서 `i+j`가 가질 수 있는 값의 범위는? 그래서 `flag_b`는 몇 칸이 필요해?

In [ ]:
n = 4
print('=== i + j 표 ===')
print('       열(i)→')
print('      ' + ''.join(f'{i:5}' for i in range(n)))
for j in range(n):
    print(f'행 {j}  ' + ''.join(f'{i+j:5}' for i in range(n)))
print()

print('=== i+j = 3 인 칸만 표시 ===')
for j in range(n):
    print('  ' + ' '.join('●' if i+j == 3 else '·' for i in range(n)))
print()
print(f'i+j 범위: 0 ~ {2*(n-1)}  →  flag_b 크기 = {2*n-1}칸')


### 5. 🟡 [손/빈칸] 반대 방향 대각선

이번엔 `i - j + 3` (4×4니까 `+3`)을 채워봐.

```
        열(i)→
        0    1    2    3
  0   [ 3 ][ 4 ][__ ][__ ]
  1   [ 2 ][__ ][__ ][__ ]
  2   [__ ][__ ][__ ][__ ]
  3   [ 0 ][__ ][__ ][ 3 ]
  ↑행(j)
```

- 같은 숫자끼리 이으면 어떤 방향이야?
- **왜 `+3`을 더해?** 안 더하면 무슨 문제가 생겨?

In [ ]:
n = 4
print('=== i - j (보정 전) ===')
print('      ' + ''.join(f'{i:5}' for i in range(n)))
for j in range(n):
    print(f'행 {j}  ' + ''.join(f'{i-j:5}' for i in range(n)))
print()
print(f'범위: {-(n-1)} ~ {n-1}  ← 음수가 있어서 배열 인덱스로 못 씀!')
print()

print(f'=== i - j + {n-1} (보정 후) ===')
print('      ' + ''.join(f'{i:5}' for i in range(n)))
for j in range(n):
    print(f'행 {j}  ' + ''.join(f'{i-j+n-1:5}' for i in range(n)))
print()
print(f'범위: 0 ~ {2*(n-1)}  →  flag_c 크기 = {2*n-1}칸  ✓')


---
# 🧠 2부 — 재귀 과정 이해하기

이제 손으로 그린 감각을 **알고리즘의 흐름**으로 옮겨보자. 아직 코드는 안 짜.

### 6. 🟡 [손] `flag` 켜고 끄기 손으로 추적 🔥🔥

**오늘의 핵심 문제.** 4×4 보드에서 `set(0)`부터 시작해 손으로 따라가 봐.
(대각선은 잠깐 무시하고 **행 중복만** 체크하는 `8queen_bb.py` 버전으로)

| 단계 | 현재 | 하는 일 | flag 상태 |
|---|---|---|---|
| 1 | `set(0)`, j=0 | flag[0]이 False → 0열 0행에 배치 | `[F,F,F,F]` |
| 2 | | `flag[0] = True` | `[T,F,F,F]` |
| 3 | `set(1)` 호출, j=0 | flag[0]이 **①____** → 건너뜀 | `[T,F,F,F]` |
| 4 | `set(1)`, j=1 | flag[1]이 **②____** → 1열 1행에 배치 | `[T,F,F,F]` |
| 5 | | `flag[1] = True` | **③____** |
| 6 | `set(2)` 호출, j=0 | flag[0]이 True → 건너뜀 | |
| 7 | `set(2)`, j=1 | flag[1]이 **④____** → 건너뜀 | |
| 8 | `set(2)`, j=2 | → 2열 2행 배치, `flag[2]=True` | **⑤____** |

*(①~⑤ 채우고 아래 실행해서 확인)*

In [ ]:
def trace_flag(n=4, max_lines=40):
    '''flag가 켜지고 꺼지는 과정을 추적 (행 중복만 체크)'''
    pos = [0] * n
    flag = [False] * n
    printed = [0]
    def fs():
        return '[' + ','.join('T' if f else 'F' for f in flag) + ']'
    def set_(i, depth=0):
        if printed[0] > max_lines:
            return
        ind = '  ' * depth
        for j in range(n):
            if printed[0] > max_lines:
                return
            if not flag[j]:
                pos[i] = j
                printed[0] += 1
                print(f'{ind}{i}열 j={j}: 비어있음 → pos[{i}]={j} 배치')
                if i == n - 1:
                    printed[0] += 1
                    print(f'{ind}  ★ 완성! pos={pos}')
                else:
                    flag[j] = True
                    printed[0] += 1
                    print(f'{ind}  flag[{j}]=True   flag={fs()}')
                    set_(i + 1, depth + 1)
                    flag[j] = False
                    printed[0] += 1
                    print(f'{ind}  flag[{j}]=False 반납  flag={fs()}')
            else:
                printed[0] += 1
                print(f'{ind}{i}열 j={j}: 이미 사용중 → 건너뜀')
    set_(0)
    print(f'\n(처음 {max_lines}줄만 표시)')

trace_flag(4, 30)


### 7. 🟡 [설명] `flag[j] = False`, 타이밍이 왜 안전한가 🔥🔥

**오늘 제일 좋은 질문이었어.** 이런 걱정이 들 수 있어:

> "`flag[j] = False`로 되돌리면, 다음 열이 볼 때 그 행이 안 쓴 걸로 기억되는 거 아니야?"

아래 실험으로 직접 확인해봐.

In [ ]:
n = 4
pos = [0] * n
flag = [False] * n

def set_check(i):
    for j in range(n):
        if not flag[j]:
            pos[i] = j
            if i == n - 1:
                pass
            else:
                flag[j] = True
                if i == 1 and j == 1:
                    print(f'  [1열이 1행 예약] flag = {flag}')
                    print(f'   ↓ 이제 set(2) 호출 — 2열이 실행되는 "동안"')
                set_check(i + 1)
                if i == 1 and j == 1:
                    print(f'   ↑ set(2) 완전히 끝나고 돌아옴')
                    print(f'  [이제서야 반납] flag[1] = False\n')
                flag[j] = False

set_check(0)
print('핵심: True와 False 사이에 set(i+1)이 통째로 들어있다!')


**(a)** `flag[1] = True`와 `flag[1] = False` 사이에 무엇이 실행돼?

**(b)** 2열(`set(2)`)이 **살아서 실행되는 동안**, `flag[1]`의 값은 뭐야?

**(c)** 그럼 "2열이 1열의 배치를 잘못 기억하는" 순간이 존재할 수 있어? 왜 그런지 **11일차 스택**과 연결해서 설명해봐.

*(여기에 답 작성)*

---
### 8. 🟡 [설명] `set(20)`은 왜 에러가 났을까

오늘 `set(20)`을 넣었더니 `IndexError`가 났지. 근데 **왜 함수 호출 자체는 됐을까?**

In [ ]:
def demo(i):
    print(f'  함수 시작! i={i}로 호출됨 — 여기까진 문제 없음')
    pos = [0] * 8
    pos[i] = 99      # 실제로 배열에 접근하는 순간
    print('  여기까지 오면 성공')

print('=== set(20) 흉내 ===')
try:
    demo(20)
except IndexError as e:
    print(f'  IndexError: {e}')
    print('  ← 호출은 성공했는데, 배열에 접근할 때 터졌다!')


**(a)** "함수 시작!" 메시지가 출력됐다는 건 무슨 뜻이야? 호출 자체는 성공했어, 실패했어?

**(b)** `def set(i: int)`라는 타입 힌트가 있는데도 왜 20을 막지 못했을까? (13~14일차 어노테이션 복습)

**(c)** `set(i)`의 `i`가 "열의 개수"가 아니라 "**지금 작업 중인 열 번호**"인 근거를 코드에서 찾아봐. 열이 8개라는 정보는 어디에 있어?

*(여기에 답 작성)*

---
### 9. 🟡 [실험] 조합이 줄어드는 3단계 🔥

한정 조건을 하나씩 추가할 때마다 탐색해야 할 조합이 얼마나 줄어드는지 직접 확인해봐.

In [ ]:
import time

print('=== 1단계: 분기만 (규칙 1 — 각 열에 1개) ===')
t = time.time()
c1 = count_stage1()
print(f'  조합 수: {c1:,}개   ({time.time()-t:.1f}초)')
print(f'  = 8^8 = {8**8:,}')
print()

print('=== 2단계: + 행 한정 (규칙 2) ===')
s2 = solve_stage2()
print(f'  조합 수: {len(s2):,}개')
print(f'  = 8! = 40,320')
print(f'  1단계 대비 {c1/len(s2):.0f}배 감소')
print()

print('=== 3단계: + 대각선 한정 (진짜 8퀸!) ===')
s3 = solve_stage3()
print(f'  조합 수: {len(s3)}개  ← 교재 정답 92개')
print(f'  2단계 대비 {len(s2)/len(s3):.0f}배 감소')
print(f'  1단계 대비 {c1/len(s3):,.0f}배 감소!')


**(a)** 1단계 → 2단계에서 조합이 약 416배 줄었어. `flag` 배열 하나 추가했을 뿐인데 왜 이렇게 큰 효과가 있을까?

**(b)** 교재 214p: "이처럼 필요하지 않은 분기를 없애서 불필요한 조합을 열거하지 않는 방법을 **한정 작업**이라고 합니다." — 한정 작업이 없으면 왜 문제를 못 푸는지, 1단계의 조합 수로 설명해봐.

*(여기에 답 작성)*

---
# 💻 3부 — 코드로 옮기기

손으로 그리고 흐름도 이해했으니, 이제 코드를 짜자.

### 10. 🟢 [빈칸] 1단계 — 분기만 구현

가장 단순한 버전부터. 각 열에 퀸 1개씩만 놓는 모든 조합을 나열해.

In [ ]:
def stage1_count(n=4):
    pos = [0] * n
    cnt = [0]
    def set_(i):
        for j in range(n):
            pos[i] = j
            if i == ___:          # TODO: 마지막 열이면?
                cnt[0] += 1
            else:
                set_(___)         # TODO: 다음 열로
    set_(0)
    return cnt[0]

for n in [3, 4, 5]:
    print(f'{n}×{n}: {stage1_count(n)}개   (기대: {n}^{n} = {n**n})')


### 11. 🟡 [빈칸] 2단계 — 행 한정 추가

`flag` 배열로 같은 행 중복을 막아봐. **True/False 짝을 놓치지 마.**

In [ ]:
def stage2_solve(n=4):
    pos = [0] * n
    flag = [False] * n
    sols = []
    def set_(i):
        for j in range(n):
            if ___ flag[j]:               # TODO: j행이 비어있으면
                pos[i] = j
                if i == n - 1:
                    sols.append(list(pos))
                else:
                    flag[j] = ___          # TODO: 예약
                    set_(i + 1)
                    flag[j] = ___          # TODO: 반납
    set_(0)
    return sols

import math
for n in [3, 4, 5]:
    s = stage2_solve(n)
    print(f'{n}×{n}: {len(s)}개   (기대: {n}! = {math.factorial(n)})')


### 12. 🔴 [빈칸] 3단계 — 대각선 한정까지 (완성!)

드디어 진짜 8퀸 문제. **대각선 인덱스 공식**을 정확히 넣어야 해.

In [ ]:
def stage3_solve(n=8):
    pos = [0] * n
    flag_a = [False] * n            # 행
    flag_b = [False] * (2*n - 1)    # / 대각선
    flag_c = [False] * (2*n - 1)    # \ 대각선
    sols = []
    def set_(i):
        for j in range(n):
            if (not flag_a[j]
                and not flag_b[___]           # TODO: / 대각선 인덱스
                and not flag_c[___]):         # TODO: \ 대각선 인덱스 (보정 포함)
                pos[i] = j
                if i == n - 1:
                    sols.append(list(pos))
                else:
                    flag_a[j] = flag_b[___] = flag_c[___] = True     # TODO
                    set_(i + 1)
                    flag_a[j] = flag_b[___] = flag_c[___] = False    # TODO
    set_(0)
    return sols

for n in [4, 5, 6, 8]:
    s = stage3_solve(n)
    print(f'{n}×{n}: {len(s):>3}개')
print()
print('8×8이 92개면 정답! 첫 번째 해:')
s8 = stage3_solve(8)
if s8:
    print(s8[0])
    draw(s8[0])


### 13. 🔴 [디버깅] 대각선 인덱스 오타 찾기 🔥

아래 코드는 오타가 하나 있어서 잘못된 답이 나와. 찾아봐.

**힌트**: 답의 개수를 세어보고, 그중 실제로 유효한 배치가 몇 개인지 검사해봐.

In [ ]:
def buggy_solve(n=8):
    pos = [0] * n
    fa = [False] * n
    fb = [False] * (2*n-1)
    fc = [False] * (2*n-1)
    sols = []
    def set_(i):
        for j in range(n):
            if (not fa[j]
                and not fb[i + 1]              # ← 여기 뭔가 이상한데?
                and not fc[i - j + (n-1)]):
                pos[i] = j
                if i == n - 1:
                    sols.append(list(pos))
                else:
                    fa[j] = fb[i+j] = fc[i-j+(n-1)] = True
                    set_(i + 1)
                    fa[j] = fb[i+j] = fc[i-j+(n-1)] = False
    set_(0)
    return sols

bad = buggy_solve(8)
print(f'버그 버전 결과: {len(bad)}개  (정답은 92개)')
valid = sum(1 for s in bad if is_valid(s)[0])
print(f'그중 실제로 유효한 배치: {valid}개')
print(f'잘못된 배치: {len(bad) - valid}개')
print()
print('잘못된 배치의 예:')
for s in bad:
    ok, why = is_valid(s)
    if not ok:
        print(f'  {s}  →  {why}')
        draw(s)
        break


**(a)** 어느 줄이 잘못됐고, 뭐로 고쳐야 해?

**(b)** 그 오타가 **왜** 대각선 검사를 무력화시키는지 설명해봐. (`i+1`은 `j`와 무관하다는 점에 주목)

*(여기에 답 작성)*

---
### 14. 🔴 [구현] 체스판으로 예쁘게 출력하기

`8queen2.py`처럼 `♛`와 `□`로 출력하는 `put()` 함수를 만들어봐.

**주의**: `pos[i] = j`는 "i**열** j**행**"이야. 화면에 출력할 땐 **행 단위로 한 줄씩** 찍어야 하니까 루프 순서를 잘 생각해.

In [ ]:
def put_board(pos):
    '''pos 배열을 체스판으로 출력'''
    n = len(pos)
    for ___ in range(n):          # TODO: 바깥 루프는 행? 열?
        for ___ in range(n):      # TODO: 안쪽 루프는?
            print('♛' if pos[i] == j else '□', end='')
        print()
    print()

# 검증
test = [0, 4, 7, 5, 2, 6, 1, 3]   # 교재 그림 5-13
put_board(test)
print('교재 216p 실행결과 첫 번째와 같아야 함')


---
## ✅ 정답 & 해설

### R-1. 하노이와 8퀸의 공통점
- ① **원반 1개** (맨 아래 원반을 옮기고, 나머지 그룹은 재귀에 맡김)
- ② **열 1개** (한 열에 퀸 1개를 놓고, 나머지 열은 재귀에 맡김)
- ③ **`set(i+1)`**
- ④ **`i == 7`** (마지막 열까지 다 놓았으면 완성)

**공통 전략**: 둘 다 "**지금 한 단계만 확실히 처리하고, 나머지는 똑같은 함수에게 맡긴다**". 이게 분할 정복법.

---
### R-2. 8퀸의 믿음의 도약
- `set(i+1)` 안에서 무슨 일이 일어나는지 **몰라도 돼.** "i+1열부터 마지막 열까지 알아서 다 채워줄 것"이라고 믿고 짜면 돼.
- `set(i)`가 책임지는 범위는 딱 **"i열에 퀸 하나를 놓는 것"**뿐이야. 그 이후(i+1열~7열)는 전부 재귀 호출의 몫.

이게 재귀의 힘이야 — 8개 열을 전부 신경 쓰는 대신 **한 열만** 생각하면 되니까 코드가 짧아져.

---
### 1. 퀸의 공격 범위
가로(같은 행) + 세로(같은 열) + 양방향 대각선. 3열 3행 퀸이면 3행 전체, 3열 전체, 그리고 `i-j`가 0인 `\` 대각선과 `i+j`가 6인 `/` 대각선 전부.

**8퀸 문제가 어려운 이유**: 퀸 하나가 판의 **상당 부분**을 커버해서, 8개를 다 놓을 자리가 매우 제한적이야.

---
### 2. 3×3에는 답이 없다
`3×3` 결과는 **0가지**. 
**왜**: 3열에 각각 1개씩 놓아야 하는데, 행도 3개뿐이라 서로 다른 행을 써야 해(3! = 6가지 후보). 그 6가지가 전부 대각선에서 걸려. 판이 작을수록 대각선 제약이 상대적으로 강해서, **2×2, 3×3은 해가 없고 4×4부터** 답이 생겨.

---
### 3. 4×4의 해
**2가지**: `[1, 3, 0, 2]`와 `[2, 0, 3, 1]`. (서로 좌우 대칭)

---
### 4. `i+j` 표 정답
```
        0    1    2    3
  0     0    1    2    3
  1     1    2    3    4
  2     2    3    4    5
  3     3    4    5    6
```
- 같은 숫자를 이으면 **`/` 방향** 대각선 (왼쪽 아래 → 오른쪽 위)
- 범위 **0~6**, 그래서 `flag_b`는 **7칸** (4×4 기준). 8×8이면 0~14로 **15칸**.
- 일반식: `2n-1`칸

---
### 5. `i-j+3` 표 정답
```
        0    1    2    3
  0     3    4    5    6
  1     2    3    4    5
  2     1    2    3    4
  3     0    1    2    3
```
- 같은 숫자를 이으면 **`\` 방향** 대각선 (왼쪽 위 → 오른쪽 아래)
- **`+3`을 더하는 이유**: 보정 전 `i-j`는 **−3 ~ 3** 범위라 **음수**가 나와. 파이썬에서 `flag_c[-2]`는 "뒤에서 2번째"라는 **완전히 다른 의미**로 해석돼서 엉뚱한 칸을 검사하게 돼. `+3`(일반적으로 `+(n-1)`)을 더하면 **0 ~ 6**으로 밀려서 안전하게 인덱스로 쓸 수 있어.

---
### 6. flag 추적 정답
- ① **True** (0행은 0열이 쓰는 중)
- ② **False** (1행은 아직 비어있음)
- ③ **`[T,T,F,F]`**
- ④ **True** (1행은 1열이 쓰는 중)
- ⑤ **`[T,T,T,F]`**

---
### 7. `flag[j] = False` 타이밍 🔥🔥

**(a)** `set(i+1)`이 **통째로** 들어있어. 즉 i+1열부터 마지막 열까지의 **모든 탐색**이 그 사이에서 일어나.

**(b)** 계속 **`True`**야. `set(2)`가 실행 중이라는 건 아직 `set(1)`의 그 줄이 안 끝났다는 뜻이니까, 다음 줄인 `flag[1] = False`는 아직 실행되지 않았어.

**(c)** **그런 순간은 존재할 수 없어.** 이유는 **11일차 스택(LIFO)**:
- 함수 호출은 스택에 쌓이고, `set(2)`가 **완전히 끝나서 스택에서 빠질 때까지** `set(1)`은 다음 줄로 못 넘어가.
- 그러니 "더 깊은 열이 실행 중" ↔ "그 열을 있게 해준 얕은 열의 flag가 True" 가 **항상 정확히 겹쳐.**
- `flag[j] = False`는 그 행에 의존하던 **모든 하위 탐색이 끝난 뒤**에만 실행돼서, 그때는 이미 그 행을 "쓰던" 열이 다른 행으로 넘어가려는 시점이라 반납이 정확해.

이 패턴 이름이 **백트래킹**: 시도 → 끝까지 탐색 → 원상복구 → 다음 시도.

---
### 8. `set(20)` 에러

**(a)** 호출 자체는 **성공**했어. 파이썬 입장에서 `set(20)`은 그냥 "정수 하나를 매개변수로 넘긴" 정상적인 호출이야. 실패는 함수 **몸통 안에서 `pos[20]`에 실제로 접근하려는 순간** 일어나.

**(b)** 타입 힌트(`i: int`)는 **강제되지 않는 메모**야 (13~14일차에 계속 나온 얘기). 게다가 `20`은 실제로 **int가 맞아서** 타입상으론 아무 문제도 없어. "0~7 사이여야 한다"는 **범위 조건은 어디에도 안 적혀 있어.**

**(c)** 열이 8개라는 정보는 세 군데:
- `pos = [0] * 8` — 배열 크기
- `for j in range(8)` — 각 열에서 시도할 행 범위
- `if i == 7:` — 마지막 열 판정

`set(i)`의 `i`는 **"지금 채우는 중인 열 번호"**야. 정상 시작은 항상 `set(0)` — "0번째 열부터 시작해라".

---
### 9. 조합 감소 3단계

**결과**: 16,777,216 → 40,320 → **92**

**(a)** `flag` 하나로 **가지 전체를 통째로 쳐내기** 때문이야. "0열이 0행을 썼다"면 1열에서 0행을 시도하는 순간 **그 아래 딸린 8⁶개의 조합을 전부 안 봐도 돼.** 한 번의 `if`로 수천~수만 개의 경우를 한꺼번에 버리는 셈. (교재 214p: "262,144가지 조합을 모두 생략할 수 있습니다")

**(b)** 1단계는 1,677만 가지를 **전부 나열**해야 하는데, 각 조합이 유효한지 검사까지 하면 현실적으로 너무 오래 걸려. 교재 205p도 "이 조합을 모두 나열하고 각 조합이 8퀸 문제의 조건을 만족하는지 조사하는 것은 **비현실적**"이라고 해. 한정 작업은 **"애초에 가능성 없는 가지에 발도 들이지 않는"** 전략이라, 답에 도달하는 시간을 실용적인 수준으로 끌어내려.

---
### 10. 1단계 정답
```python
if i == n - 1:        # 마지막 열
    cnt[0] += 1
else:
    set_(i + 1)       # 다음 열
```

---
### 11. 2단계 정답
```python
if not flag[j]:       # j행이 비어있으면
    ...
    flag[j] = True    # 예약
    set_(i + 1)
    flag[j] = False   # 반납
```

---
### 12. 3단계 정답
```python
if (not flag_a[j]
    and not flag_b[i + j]
    and not flag_c[i - j + (n-1)]):
    pos[i] = j
    if i == n - 1:
        sols.append(list(pos))
    else:
        flag_a[j] = flag_b[i + j] = flag_c[i - j + (n-1)] = True
        set_(i + 1)
        flag_a[j] = flag_b[i + j] = flag_c[i - j + (n-1)] = False
```
**n별 해의 개수**: 4×4=2, 5×5=10, 6×6=4, 8×8=**92** ✓

---
### 13. 디버깅 정답 🔥

**(a)** `not fb[i + 1]` → **`not fb[i + j]`**

**(b) 왜 무력화되나**: `i+1`은 **`j`와 아무 상관이 없어.** `for j in range(8)` 루프를 도는 내내 `i+1`은 **똑같은 값**이라, "지금 시도 중인 (i열, j행) 칸이 어느 대각선에 속하는지"를 전혀 반영하지 못해. 
반면 `i+j`는 7번 개념에서 봤듯이 **그 칸이 속한 `/` 대각선의 고유 번호**야. 이걸 `i+1`로 바꾸면 `/` 방향 대각선 검사가 사실상 사라져서, 결과가 92개가 아니라 **162개**로 부풀고 그중 대부분이 **실제로는 서로 공격 가능한 잘못된 배치**가 돼.

> 💡 무서운 점: 에러가 안 나고 **그럴듯한 답이 나온다는 것.** 13일차 `factorial(-1)`이 조용히 1을 반환한 것, 9일차 오픈해시 `None` 버그와 같은 종류의 **조용한 실패**야.

---
### 14. 체스판 출력 정답
```python
def put_board(pos):
    n = len(pos)
    for j in range(n):            # 바깥 = 행 (한 줄씩 찍으니까)
        for i in range(n):        # 안쪽 = 열 (왼→오른쪽으로)
            print('♛' if pos[i] == j else '□', end='')
        print()
    print()
```

**왜 바깥이 행인가**: 화면 출력은 **위에서 아래로 한 줄씩** 나가. 한 줄 = 하나의 **행**이니까 바깥 루프가 행(`j`)이어야 해. 안쪽 루프가 열(`i`)을 돌면서 그 행에 퀸이 있는지(`pos[i] == j`) 확인하는 거야.

**흔한 실수**: `pos[j] == i`로 쓰면 판이 **대각선 대칭으로 뒤집혀서** 출력돼. `pos`의 인덱스는 항상 **열**이라는 걸 기억!

---

## 📌 핵심 3줄 요약

1. **행 = 가로 팀(번호는 위→아래), 열 = 세로 팀(번호는 왼→오른쪽).** `pos[i] = j`는 "**i열**에 놓은 퀸이 **j행**에 있다"는 뜻.
2. **분기(모든 경우 나열) + 한정(불가능한 가지 미리 쳐내기) = 분기 한정법.** `flag`로 한정하면 1,677만 → 4만 → **92**로 줄어든다.
3. **`flag[j] = True → set(i+1) → flag[j] = False`가 백트래킹.** True와 False 사이에 하위 탐색이 통째로 들어있어서, 반납 타이밍이 **자동으로** 안전하다(11일차 스택 LIFO).

## 🗂️ 스터디 진행 가이드
- **1부 (1~5번)**: 전원 필수. **손으로 안 그리면 절대 이해 안 됨**
  - 특히 4·5번 대각선 표는 `i+j`, `i-j+7`이 왜 그 공식인지의 전부
- **2부 (6~9번)**: 팀 목표선. 오늘 나온 질문들이 그대로 문제로
  - **7번이 오늘 최고의 질문** 🔥 — "flag 반납하면 다음 열이 잘못 기억하는 거 아냐?"
  - 8번 `set(20)`, 9번 조합 감소 실측
- **3부 (10~14번)**: 코딩. 1→2→3단계를 순서대로 쌓아 올리기
  - **13번 디버깅은 꼭** — `i+1` vs `i+j`, 조용한 실패의 무서움
- **금요일 코딩테스트 범위**: 1부 + 2부 (1~9번)

## 🔗 05장 재귀 알고리즘 완주 총정리

| | factorial (13일) | recur (14일) | 하노이 (15일) | **8퀸 (16일)** |
|---|---|---|---|---|
| 재귀 호출 | 1번 | 2번 | 2번 | **for문 안에서 n번** |
| 구조 | 직선 | 트리(2갈래) | 트리(2갈래) | **트리(8갈래)** |
| 복잡도 | O(n) | O(1.618ⁿ) | O(2ⁿ) | O(n!) → 한정으로 대폭 감소 |
| 핵심 기법 | 기저 조건 | 하향식/상향식 분석 | 분할 정복 | **분할 정복 + 백트래킹** |

### 🎓 오늘 회수된 개념들
- **15일차 분할 정복 / 믿음의 도약** → `set(i+1)`에 나머지 열을 맡김 (R-1, R-2)
- **11일차 스택 LIFO** → `flag` 반납 타이밍이 안전한 이유 (7번)
- **13~14일차 어노테이션** → `i: int`가 범위를 막지 못함 (8번)
- **13일차 조용한 실패** → `i+1` 오타가 에러 없이 틀린 답 (13번)
- **3일차 음수 인덱스** → `flag_c`에 `+7`을 더하는 이유 (5번)

> **다음**: 06장 정렬 알고리즘! 재귀가 병합 정렬·퀵 정렬에서 다시 등장해.
